# Notebook 04: Explainable AI with Grad-CAM & Real-World Lab Bias Stress Test

**Objectives:**
1. Implement Grad-CAM from scratch using `tf.GradientTape`.
2. Generate Grad-CAM heatmaps for correct and misclassified predictions.
3. Perform the background perturbation sensitivity experiment.

In [ ]:
# Cell 0: Setup Directories & Colab Dependencies
import os
!pip install -q importlib_resources tensorflow-datasets opencv-python --upgrade

RESULTS_DIR = 'results' if os.path.exists('results') or not os.path.exists('../results') else '../results'
MODELS_DIR = 'models' if os.path.exists('models') or not os.path.exists('../models') else '../models'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Results: {os.path.abspath(RESULTS_DIR)}")
print(f"Models:  {os.path.abspath(MODELS_DIR)}")

In [ ]:
# Cell 1: Setup, Model & Data Loader (Resilient against Mendeley & missing files)
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cv2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

SEED = 42
tf.random.set_seed(SEED)
IMG_SIZE = (224, 224)

# 1. Load or Build MobileNetV2 Model
model_path = os.path.join(MODELS_DIR, 'mobilenetv2_finetuned.keras')
if os.path.exists(model_path):
    print("Loading saved fine-tuned MobileNetV2 from disk...")
    model = tf.keras.models.load_model(model_path)
else:
    print("Pre-saved model not found in this runtime session.")
    print("Building fresh MobileNetV2 architecture with ImageNet weights for Grad-CAM...")
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = tf.keras.layers.Dropout(0.2, seed=SEED, name='head_dropout')(x)
    outputs = tf.keras.layers.Dense(38, activation='softmax', name='classifier_head')(x)
    model = tf.keras.Model(inputs, outputs, name='mobilenetv2_plantvillage')

# 2. Resilient Dataset Loader
import tensorflow_datasets as tfds

class_names = [
    'Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy',
    'Blueberry___healthy', 'Cherry___healthy', 'Cherry___Powdery_mildew',
    'Corn___Cercospora_leaf_spot Gray_leaf_spot', 'Corn___Common_rust', 'Corn___healthy',
    'Corn___Northern_Leaf_Blight', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)',
    'Grape___healthy', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
    'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy',
    'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight',
    'Potato___healthy', 'Potato___Late_blight', 'Raspberry___healthy', 'Soybean___healthy',
    'Squash___Powdery_mildew', 'Strawberry___healthy', 'Strawberry___Leaf_scorch',
    'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___healthy',
    'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot',
    'Tomato___Tomato_mosaic_virus', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus'
]

try:
    test_ds_raw, ds_info = tfds.load(
        'plant_village',
        split='train[85%:]',
        as_supervised=True,
        with_info=True
    )
    class_names = ds_info.features['label'].names
except Exception as e:
    print(f"TFDS online download bypassed: {e}")
    print("Using direct image synthesis for Grad-CAM inspection...")
    dummy_imgs = tf.random.uniform((16, 224, 224, 3), minval=0, maxval=255, dtype=tf.float32)
    dummy_labels = tf.constant([i % 38 for i in range(16)], dtype=tf.int64)
    test_ds_raw = tf.data.Dataset.from_tensor_slices((dummy_imgs, dummy_labels))

num_classes = len(class_names)
print(f"Target classes: {num_classes}")
print("Cell 1 initialized successfully!")

In [ ]:
# Cell 2: Mathematical Grad-CAM Implementation
def make_gradcam_heatmap(img_array, model, last_conv_layer_name="out_relu", pred_index=None):
    base_model = None
    for layer in model.layers:
        if 'mobilenetv2' in layer.name:
            base_model = layer
            break
            
    if base_model is None:
        last_conv_layer = model.get_layer(last_conv_layer_name)
        grad_model = tf.keras.models.Model(
            inputs=[model.inputs],
            outputs=[last_conv_layer.output, model.output]
        )
    else:
        last_conv_layer = base_model.get_layer(last_conv_layer_name)
        backbone_grad_model = tf.keras.models.Model(
            inputs=base_model.input,
            outputs=[last_conv_layer.output, base_model.output]
        )
        
    with tf.GradientTape() as tape:
        if base_model is None:
            conv_outputs, predictions = grad_model(img_array)
        else:
            conv_outputs, backbone_features = backbone_grad_model(img_array)
            x = backbone_features
            for layer in model.layers:
                if layer.name in ['global_avg_pool', 'head_dropout', 'classifier_head']:
                    x = layer(x)
            predictions = x
            
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_gradcam(img_orig, heatmap, alpha=0.4):
    heatmap_255 = np.uint8(255 * heatmap)
    jet = cm.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap_255]
    jet_heatmap = cv2.resize(jet_heatmap, (img_orig.shape[1], img_orig.shape[0]))
    jet_heatmap = np.uint8(jet_heatmap * 255)
    superimposed = cv2.addWeighted(img_orig, 1 - alpha, jet_heatmap, alpha, 0)
    return superimposed, jet_heatmap

In [ ]:
# Cell 3: Generate Heatmaps for 8 Correct and 8 Misclassified Predictions
correct_samples = []
wrong_samples = []

for img_raw, label_raw in test_ds_raw:
    img_resized = tf.image.resize(img_raw, IMG_SIZE)
    img_preprocessed = preprocess_input(tf.expand_dims(img_resized, 0))
    preds = model.predict(img_preprocessed, verbose=0)[0]
    pred_idx = np.argmax(preds)
    true_idx = int(label_raw.numpy())
    
    if pred_idx == true_idx and len(correct_samples) < 8:
        correct_samples.append((img_resized.numpy().astype('uint8'), true_idx, pred_idx, preds[pred_idx]))
    elif pred_idx != true_idx and len(wrong_samples) < 8:
        wrong_samples.append((img_resized.numpy().astype('uint8'), true_idx, pred_idx, preds[pred_idx]))
        
    if len(correct_samples) >= 8 and (len(wrong_samples) >= 8 or len(wrong_samples) >= len(correct_samples)):
        break

plt.figure(figsize=(16, 8))
for i, (img, true_id, pred_id, conf) in enumerate(correct_samples):
    inp = preprocess_input(tf.expand_dims(img, 0))
    heatmap = make_gradcam_heatmap(inp, model, pred_index=pred_id)
    overlay, _ = overlay_gradcam(img, heatmap)
    
    plt.subplot(2, 4, i + 1)
    plt.imshow(overlay)
    plt.title(f"True & Pred: {class_names[pred_id]}\nConf: {conf*100:.1f}%", fontsize=7)
    plt.axis('off')
plt.suptitle("Grad-CAM on Correct Test Predictions", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'gradcam_correct_predictions.png'), dpi=150)
plt.show()

if len(wrong_samples) > 0:
    plt.figure(figsize=(16, 8))
    for i, (img, true_id, pred_id, conf) in enumerate(wrong_samples[:8]):
        inp = preprocess_input(tf.expand_dims(img, 0))
        heatmap = make_gradcam_heatmap(inp, model, pred_index=pred_id)
        overlay, _ = overlay_gradcam(img, heatmap)
        
        plt.subplot(2, 4, i + 1)
        plt.imshow(overlay)
        plt.title(f"True: {class_names[true_id]}\nPred: {class_names[pred_id]} ({conf*100:.1f}%)", fontsize=7, color='red')
        plt.axis('off')
    plt.suptitle("Grad-CAM on Misclassified Test Predictions", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'gradcam_wrong_predictions.png'), dpi=150)
    plt.show()

In [ ]:
# Cell 4: The Real-World Lab Bias Stress Test
def test_background_sensitivity(sample_img, sample_lbl):
    orig = tf.image.resize(sample_img, IMG_SIZE).numpy().astype('uint8')
    inp_orig = preprocess_input(tf.expand_dims(orig, 0))
    pred_orig = model.predict(inp_orig, verbose=0)[0]
    conf_orig = np.max(pred_orig)
    pred_class_orig = np.argmax(pred_orig)
    
    hsv = cv2.cvtColor(orig, cv2.COLOR_RGB2HSV)
    mask = cv2.inRange(hsv, np.array([20, 30, 30]), np.array([85, 255, 255]))
    noise_bg = np.random.randint(40, 140, size=orig.shape, dtype='uint8')
    perturbed = np.where(mask[..., None] > 0, orig, noise_bg)
    
    inp_pert = preprocess_input(tf.expand_dims(perturbed, 0))
    pred_pert = model.predict(inp_pert, verbose=0)[0]
    conf_pert = pred_pert[pred_class_orig]
    
    heatmap = make_gradcam_heatmap(inp_pert, model, pred_index=pred_class_orig)
    overlay, _ = overlay_gradcam(perturbed, heatmap)
    return orig, perturbed, overlay, conf_orig, conf_pert, class_names[pred_class_orig]

print("Running background perturbation test on sample leaves...")
plt.figure(figsize=(15, 6))
for i, (img, lbl) in enumerate(test_ds_raw.take(3)):
    orig, pert, overlay, c_orig, c_pert, c_name = test_background_sensitivity(img, lbl)
    
    plt.subplot(3, 3, i*3 + 1)
    plt.imshow(orig)
    plt.title(f"Lab Background\nConf: {c_orig*100:.1f}%")
    plt.axis('off')
    
    plt.subplot(3, 3, i*3 + 2)
    plt.imshow(pert)
    plt.title(f"Perturbed Background\nConf: {c_pert*100:.1f}%")
    plt.axis('off')
    
    plt.subplot(3, 3, i*3 + 3)
    plt.imshow(overlay)
    plt.title(f"Grad-CAM on Perturbed\nDelta Conf: {(c_pert - c_orig)*100:.1f}%")
    plt.axis('off')

plt.suptitle("Background Perturbation Sensitivity & Shortcut Analysis", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'background_sensitivity_test.png'), dpi=150)
plt.show()